<a href="https://colab.research.google.com/github/Ricky-S-Gong/search-rec-ads-portfolio/blob/codex/workspace-organization/research/kuairand/training/notebooks/week1_content_retraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KuaiRand-Pure — Content Feature Retraining

Restore Yutao’s publication-content ItemTower, exclude current visibility and future monthly statistics, and encode user history with the same complete item representation. Two new runs use the same frozen requests, candidates, seed 42 and one-epoch budget as the ID-only experiment. Historical results remain in their separate release.

**Metadata limitation:** non-duration publication attributes are used under a fixed-content assumption, not proven historical versions. Duration agrees with 154,273 comparable train/validation positive records.


## 1. Mount existing shared Drive data

Add a shortcut to the team folder **kuairand-Two-Tower** under **My Drive** in the same Google account used by Colab. A “Shared with me” folder is not automatically exposed by `drive.mount`.

The data have already been prepared and published. This workflow does **not** download data from Zenodo or the Drive API. It reads the mounted files directly. `/content` is used only for code, dependencies and an optional local copy of existing Drive files.


In [ ]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
SHARED_ROOT = Path("/content/drive/MyDrive/kuairand-Two-Tower")
RUN_LABEL = "week1-pure-content-v1-001"  # Change for a genuinely new experiment.
REF = "codex/workspace-organization"
COMMIT = ""  # For an exact rerun, paste the full commit printed below.
USE_LOCAL_DATA_CACHE = False  # Optional Drive-to-/content copy; never a source re-download.
if not SHARED_ROOT.is_dir():
    raise FileNotFoundError(
        "Add the shared-folder shortcut to My Drive, or set SHARED_ROOT to its mounted path."
    )
DATA_DIR = SHARED_ROOT / "datasets/week1-pure-content-v1"
WORK_DIR = SHARED_ROOT / "runs/week1-pure" / RUN_LABEL
RELEASE_DIR = SHARED_ROOT / "releases/week1-pure" / RUN_LABEL
print("Dataset:", DATA_DIR, "\nRuns:", WORK_DIR, "\nRelease:", RELEASE_DIR)

## 2. Freeze source and recreate the locked training environment

The branch is resolved to one immutable commit before running. Resume requires the same code, config, data, Python/PyTorch versions and device type. The uv runtime is pinned to the repository requirement.


In [ ]:
def run_command(command, cwd=None):
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    with subprocess.Popen(
        list(map(str, command)),
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)


REPO_URL = "https://github.com/Ricky-S-Gong/search-rec-ads-portfolio.git"
REPO_DIR = Path("/content/sar-week1-code")
if not COMMIT:
    remote = subprocess.check_output(["git", "ls-remote", REPO_URL, "refs/heads/" + REF], text=True)
    if not remote.strip():
        raise ValueError("Set an existing REF or immutable COMMIT")
    COMMIT = remote.split()[0]
if len(COMMIT) != 40 or any(c not in "0123456789abcdef" for c in COMMIT):
    raise ValueError("Require a full commit SHA")
if not REPO_DIR.exists():
    run_command(["git", "clone", REPO_URL, REPO_DIR])
    run_command(["git", "-C", REPO_DIR, "checkout", "--detach", COMMIT])
else:
    actual = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
    ).strip()
    if actual != COMMIT:
        raise ValueError("Existing checkout differs; use a fresh runtime")
PROJECT_DIR = REPO_DIR / "research/kuairand/training"
run_command([sys.executable, "-m", "pip", "install", "uv==0.12.1"])
os.environ["UV_CACHE_DIR"] = "/content/sar-uv-cache"
os.environ["UV_PYTHON_INSTALL_DIR"] = "/content/sar-uv-python"
run_command(["uv", "sync", "--locked", "--python", "3.11.15"], cwd=PROJECT_DIR)
PYTHON = PROJECT_DIR / ".venv/bin/python"
CLI = PROJECT_DIR / "run.py"
CONFIGS = {name: PROJECT_DIR / ("configs/pure-content-" + name + ".yaml") for name in ("basic", "enhanced")}
WORK_DIR.mkdir(parents=True, exist_ok=True)
stamp = WORK_DIR / "source_commit.txt"
if stamp.exists() and stamp.read_text().strip() != COMMIT:
    raise ValueError("This run label belongs to another source commit; select a new RUN_LABEL")
stamp.write_text(COMMIT + "\n")
print("Frozen source commit:", COMMIT)
run_command([PYTHON, CLI, "check-env"], cwd=REPO_DIR)

## 3. Validate the published, mounted dataset

All source hashes and prepared file hashes are checked. There is one shared prepared manifest for both models. Re-running this cell reads existing artifacts rather than redoing preprocessing.


In [ ]:
EXPECTED_MANIFEST_SHA256 = "1f573d11966024fc42b3ee9b78bd71663fde7b65c56743902384a7bd31376bac"
if not (DATA_DIR / "manifest.json").is_file():
    raise FileNotFoundError(
        "Prepared dataset missing from shared Drive; do not download a second copy of the raw data."
    )
if (
    hashlib.sha256((DATA_DIR / "manifest.json").read_bytes()).hexdigest()
    != EXPECTED_MANIFEST_SHA256
):
    raise ValueError("Published manifest differs; re-audit before training")
if USE_LOCAL_DATA_CACHE:
    import shutil

    cache = Path("/content/week1-pure-completion-v1")
    cache.mkdir(exist_ok=True)
    for source in DATA_DIR.iterdir():
        if source.is_file():
            target = cache / source.name
            if not target.exists():
                shutil.copy2(source, target)
            if (
                hashlib.sha256(target.read_bytes()).hexdigest()
                != hashlib.sha256(source.read_bytes()).hexdigest()
            ):
                raise ValueError("Local cache mismatch: " + source.name)
    DATA_DIR = cache
MANIFEST = DATA_DIR / "manifest.json"
manifest = json.loads(MANIFEST.read_text())
audit = json.loads((DATA_DIR / "audit.json").read_text())
for name, config in CONFIGS.items():
    run_command([PYTHON, CLI, "check-config", "--config", config], cwd=REPO_DIR)
    run_command(
        [PYTHON, CLI, "check-data", "--config", config, "--manifest", MANIFEST], cwd=REPO_DIR
    )

## 4. Inspect EDA, feature values and temporal limitations

These statistics use warmup/train/validation only. The dataset retains real repeat views, removes only completely identical raw rows, and uses official `long_view` without relabeling.

The source `date` does not consistently match a fixed UTC or UTC+08 calendar; the split explicitly uses Unix UTC boundaries. `available_ms = time_ms + play_time_ms` is a conservative proxy, not an independently logged completion timestamp. Upload date is a catalog eligibility proxy, not a full visibility snapshot.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

display(pd.DataFrame(audit["splits"]).T)
print("Exact duplicate rows removed:", audit["exact_duplicates_removed"])
print("Histories changed by completion timing:", audit["request_prior_histories_changed"])
print("Date mismatch counts:", audit["recorded_date_mismatches"])
print("Candidate videos:", audit["candidate_count"])
print("Excluded snapshot-only item features:", audit["excluded_snapshot_features"])
print("Limitations:", *audit["limitations"], sep="\n- ")
feature_rows = [{"feature": k, **v} for k, v in audit["features"].items()]
display(pd.DataFrame(feature_rows)[["feature", "count", "missing", "cardinality", "examples"]])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(range(51), audit["history_length_histogram"])
axes[0].set(xlabel="Available positive history length", ylabel="Requests")
pd.DataFrame(audit["splits"]).T["positive_rate"].plot.bar(ax=axes[1], ylim=(0, 1), legend=False)
axes[1].set(ylabel="Official long_view positive rate")
plt.tight_layout()
plt.show()
fitted_items = set(manifest["item_encoding"]["fit_item_ids"])
validation_items = pd.read_parquet(DATA_DIR / "validation.parquet", columns=["item_id"])["item_id"]
unknown_catalog_rate = sum(row[0] == 0 for row in manifest["item_encoding"]["categorical"]) / len(
    manifest["item_ids"]
)
print("Unknown item-category rate in catalog:", unknown_catalog_rate)
print(
    "Unknown item-category rate among validation targets:",
    (~validation_items.isin(fitted_items)).mean(),
)

## 5. Model inputs

**Both item towers:** seven categorical fields (item_id, author_id, video_type, upload_type, music_id, music_type, tag), each embedded to 8 values, plus four train-imputed/standardized numbers (video_duration, server_width, server_height, upload_day): 60→64→64, then L2. visible_status and whole-month statistics are excluded.

**Basic user tower:** user ID 32 + tab 8 + cyclical time 4 = 44→128→64→64. **Enhanced:** add a 64-dimensional masked mean of up to 50 available history items encoded with the complete content item tower; 108→128→64→64. User profile counts/status have no historical version and are excluded.

Both share positive-only long_view supervision, duplicate-masked in-batch cross-entropy, Adam lr=0.001, batch=256, one epoch, seed=42, temperature=0.1 and CUDA AMP. New W&B runs and Drive directories are independent of the old experiments.


## 6. GPU and W&B preflight

Enable the notebook's access to the `WANDB_API_KEY` Colab Secret. The key stays in the process environment. This cell verifies both models without an optimizer update or a setup-only W&B run.


In [ ]:
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
for config in CONFIGS.values():
    run_command(
        [PYTHON, CLI, "preflight", "--config", config, "--manifest", MANIFEST], cwd=REPO_DIR
    )

## 7. Train or resume the two fair baseline experiments

A complete matching run is reused. An interrupted run resumes from its latest intact checkpoint in the same directory. A config/code/data mismatch fails rather than silently starting over. Historical runs remain untouched.


In [ ]:
CHECKPOINTS = {}
for name, config in CONFIGS.items():
    run_dir = WORK_DIR / name
    publication = run_dir / "run_manifest.json"
    complete = publication.exists() and json.loads(publication.read_text())["complete"]
    if not complete:
        command = [
            PYTHON,
            CLI,
            "train",
            "--config",
            config,
            "--manifest",
            MANIFEST,
            "--output",
            run_dir,
        ]
        if publication.exists():
            command.append("--resume-latest")
        run_command(command, cwd=REPO_DIR)
    saved = json.loads(publication.read_text())
    if not saved["complete"] or saved["consumed"] != saved["target_samples"]:
        raise ValueError("Incomplete sample traversal: " + name)
    CHECKPOINTS[name] = Path(saved["checkpoint"])
    print(
        name,
        "run:",
        saved["run_id"],
        "samples:",
        saved["consumed"],
        "checkpoint:",
        CHECKPOINTS[name],
    )

## 8. Plot measured training and validation losses

Curves are loaded from local metrics logs, not typed numbers. Hardware peak memory and per-step timing are retained in the same logs and W&B.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for name in CONFIGS:
    records = [
        json.loads(line) for line in (WORK_DIR / name / "metrics.jsonl").read_text().splitlines()
    ]
    train_rows = [r for r in records if "train/loss" in r]
    ax.plot(
        [r["train_step"] for r in train_rows],
        [r["train/loss"] for r in train_rows],
        label=name,
        alpha=0.65,
    )
    print(name, "validation:", [r for r in records if "validation/inbatch_loss" in r])
ax.set(xlabel="Optimizer step", ylabel="In-batch cross entropy")
ax.legend()
plt.show()

## 9. Export and verify the complete candidate catalog

The exporter preserves original item IDs and binds vectors to their checkpoint. Evaluation validates each shard checksum, every candidate ID, the 64-dimensional shape and unit norms.


In [ ]:
VECTOR_DIRS = {}
start_id, end_id = min(manifest["item_ids"]), max(manifest["item_ids"]) + 1
for name, config in CONFIGS.items():
    vector_dir = WORK_DIR / name / "vectors"
    if not (vector_dir / "manifest.json").is_file():
        run_command(
            [
                PYTHON,
                CLI,
                "export",
                "--config",
                config,
                "--manifest",
                MANIFEST,
                "--checkpoint",
                CHECKPOINTS[name],
                "--output",
                vector_dir,
                "--start-id",
                start_id,
                "--end-id",
                end_id,
            ],
            cwd=REPO_DIR,
        )
    VECTOR_DIRS[name] = vector_dir

## 10. Evaluate popularity, basic and enhanced models

Use every retained nonrandom validation-positive request against the full time-eligible catalog. Repeats are allowed. Rankings sort by score descending then raw item ID ascending. Unavailable targets remain in the denominator. This is validation, not the sealed final test.

Timing uses one warmup batch; reports distinguish batched latency from individual-request latency and include feature assembly/user encoding/CPU exact retrieval.


In [ ]:
RESULTS = []
for name in ("popularity", "basic", "enhanced"):
    evaluation_dir = WORK_DIR / ("evaluation-" + name)
    result = evaluation_dir / "evaluation.json"
    config = CONFIGS["basic" if name == "popularity" else name]
    if not result.is_file():
        command = [
            PYTHON,
            CLI,
            "evaluate",
            "--config",
            config,
            "--manifest",
            MANIFEST,
            "--output",
            evaluation_dir,
            "--label",
            name,
        ]
        if name != "popularity":
            command += ["--checkpoint", CHECKPOINTS[name], "--vectors", VECTOR_DIRS[name]]
        run_command(command, cwd=REPO_DIR)
    RESULTS.append(result)
display(
    pd.DataFrame(
        [
            {"model": json.loads(p.read_text())["model"], **json.loads(p.read_text())["metrics"]}
            for p in RESULTS
        ]
    )
)

## 11. Verify real GPU interruption and recovery

The enhanced configuration is rerun in a separate verification directory, intentionally interrupted at step 25 and resumed. Final parameters, optimizer, AMP scaler, RNG state and sample cursor must exactly match the uninterrupted reference. This produces a second W&B run solely as recovery evidence.


In [ ]:
RECOVERY_DIR = WORK_DIR / "enhanced-recovery-verification"
RESUME_CHECK = RECOVERY_DIR / "resume_verification.json"
if not RESUME_CHECK.is_file():
    run_command(
        [
            PYTHON,
            CLI,
            "verify-resume",
            "--config",
            CONFIGS["enhanced"],
            "--manifest",
            MANIFEST,
            "--checkpoint",
            CHECKPOINTS["enhanced"],
            "--output",
            RECOVERY_DIR,
        ],
        cwd=REPO_DIR,
    )
print(json.loads(RESUME_CHECK.read_text()))

## 12. Publish the stage release and inspect completion status

A stage is complete only with matching popularity/basic/enhanced results and CUDA recovery evidence. Missing results stay explicit. The release contains aggregate JSON/Markdown; request-level predictions and model files stay under the private run directories.


In [ ]:
if not (RELEASE_DIR / "stage_report.json").is_file():
    run_command(
        [
            PYTHON,
            CLI,
            "stage-report",
            "--config",
            CONFIGS["basic"],
            "--results",
            *RESULTS,
            "--resume-check",
            RESUME_CHECK,
            "--output",
            RELEASE_DIR,
        ],
        cwd=REPO_DIR,
    )
(RELEASE_DIR / "source_commit.txt").write_text(COMMIT + "\n")
report = json.loads((RELEASE_DIR / "stage_report.json").read_text())
print("Stage complete:", report["complete"], "\nOutstanding:", report["blockers"])
print((RELEASE_DIR / "stage_report.md").read_text())
if not report["complete"]:
    raise ValueError("Week 1 acceptance is incomplete; inspect blockers")
print("Ready for the final LaTeX report. Release:", RELEASE_DIR)

## 13. Report handoff

Compare this release to the ID-only release using unchanged request, validation and candidate hashes; feature-manifest hashes intentionally differ. Do not claim verified as-of metadata or overwrite the earlier experiments. Record true measured results in the Chinese week1 report after completion.
